## Zadanie 5 – Macierz korelacji  
Dla datasetu California Housing (generator make_california_housing() z Lekcji 12) narysuj ASCII-  
heatmap macierzy korelacji. Zidentyfikuj 3 pary cech o najwyższej korelacji (poza targetem MedHouseVal ).  

Dataset:  
df = make_california_housing(n_samples=2000)

Wymagania:  
Macierz: C = np.corrcoef(df.values.T) albo ręcznie przez pearson_corr w pętli  
ASCII-heatmap: ascii_heatmap(C, labels=df.columns) z Przykładu 1 Sekcji 2  
Lista top-3 par (i, j) po np.abs(C) (ignoruj diagonalę i target)  
Interpretacja: dlaczego te cechy mogą być skorelowane (np. Latitude / Longitude ,  
AveRooms / AveBedrms )

In [1]:
import numpy as np  # NumPy wykorzystamy do obliczenia macierzy korelacji
import pandas as pd  # Pandas wykorzystamy do utworzenia DataFrame


# Generator syntetycznego datasetu California Housing.
def make_california_housing(n_samples=2000, seed=42):
    # Tworzymy generator liczb losowych z ustalonym ziarnem.
    # Dzięki seed=42 przy każdym uruchomieniu otrzymamy takie same dane, pod warunkiem, że zresetujemy generator(ponownie wywołamy: seed=42)
    rng = np.random.default_rng(seed)

    # Tworzymy kolejne cechy opisujące rejony i znajdujące się w nich domy.
    MedInc = rng.gamma(shape=2.5, scale=1.6, size=n_samples) + 0.5
    HouseAge = rng.uniform(1, 52, size=n_samples)
    AveRooms = 3 + 0.8 * MedInc + rng.normal(0, 0.8, size=n_samples)
    AveBedrms = 1 + 0.1 * AveRooms + rng.normal(0, 0.1, size=n_samples)
    Population = rng.gamma(shape=2.0, scale=700, size=n_samples)
    AveOccup = 2.5 + rng.normal(0, 0.6, size=n_samples)
    Latitude = rng.uniform(32.5, 42.0, size=n_samples)
    Longitude = rng.uniform(-124.5, -114.0, size=n_samples)

    # Tworzymy target, czyli syntetyczną medianę wartości domu.
    MedHouseVal = (
        0.45 * MedInc
        - 0.01 * HouseAge
        + 0.03 * AveRooms
        - 0.02 * AveBedrms
        - 0.0001 * Population
        + 0.12 * (Latitude - 36)
        - 0.05 * (Longitude + 119)
        + rng.normal(0, 0.4, size=n_samples)
    )

    # Ograniczamy wartości targetu do przedziału od 0.15 do 5.0.
    MedHouseVal = np.clip(MedHouseVal, 0.15, 5.0)

    # Łączymy wszystkie tablice w jeden DataFrame i zwracamy go z funkcji.
    return pd.DataFrame({
        "MedInc": MedInc,
        "HouseAge": HouseAge,
        "AveRooms": AveRooms,
        "AveBedrms": AveBedrms,
        "Population": Population,
        "AveOccup": AveOccup,
        "Latitude": Latitude,
        "Longitude": Longitude,
        "MedHouseVal": MedHouseVal,
    })


# Generujemy 2000 wierszy danych zgodnie z treścią zadania.
df = make_california_housing(n_samples=2000)

# Wyświetlamy pierwsze 5 wierszy, aby sprawdzić strukturę danych.
df.head()

,MedInc,HouseAge,AveRooms,AveBedrms,Population,AveOccup,Latitude,Longitude,MedHouseVal
0,4.734977,40.311324,6.884812,1.606462,885.607473,1.436806,40.967157,-114.373283,1.862964
1,6.051457,26.516096,9.062208,2.065442,2715.582991,2.251636,38.671920,-120.034416,3.384592
2,4.276549,40.160726,6.771253,1.684698,1021.146138,3.267557,38.638543,-124.278583,1.891606
3,3.927248,14.661656,6.282833,1.674368,1037.793205,2.655178,40.260311,-118.824055,2.341919
4,6.477596,31.064823,8.022173,1.775457,1053.317499,1.649014,34.068856,-117.400009,2.347516


In [2]:
# Funkcja zamienia wartość korelacji na znak używany w heatmapie ASCII.
def corr_to_char(r):
    if r >= 0.75:
        return "█"
    if r >= 0.50:
        return "▓"
    if r >= 0.25:
        return "▒"
    if r >= 0.05:
        return "░"
    if r > -0.05:
        return "·"
    if r > -0.25:
        return "-"
    if r > -0.50:
        return "="
    if r > -0.75:
        return "#"
    return "@"


# Funkcja wyświetla macierz korelacji w postaci tekstowej heatmapy.
def ascii_heatmap(M, labels):
    # Skracamy nazwy kolumn do 7 znaków, aby tabela była czytelna.
    short_labels = [label[:7] for label in labels]

    # Wyświetlamy tytuł i legendę znaków.
    print("\nASCII heatmap macierzy korelacji")
    print("Legenda: █/▓/▒/░ = dodatnia, · = około 0, -/=/#/@ = ujemna")

    # Tworzymy nagłówek zawierający skrócone nazwy kolumn.
    header = " " * 11 + "".join(f"{label:>8}" for label in short_labels)
    print(header)

    # Przechodzimy po kolejnych wierszach macierzy.
    for i in range(len(labels)):
        row = f"{short_labels[i]:>10} "

        # Do wiersza dodajemy znak oraz wartość każdej korelacji.
        for j in range(len(labels)):
            row += f"{corr_to_char(M[i, j])}{M[i, j]:+5.2f}  "

        print(row)

In [3]:
# df.values pobiera z DataFrame same wartości liczbowe.
# .T zamienia wiersze z kolumnami, ponieważ corrcoef ma liczyć korelacje cech.
# Wynik zapisujemy w nowej zmiennej C - nie modyfikujemy oryginalnego df.
C = np.corrcoef(df.values.T)

# Wyświetlamy macierz za pomocą funkcji z przykładu z lekcji.
ascii_heatmap(C, labels=df.columns)


ASCII heatmap macierzy korelacji
Legenda: █/▓/▒/░ = dodatnia, · = około 0, -/=/#/@ = ujemna
             MedInc HouseAg AveRoom AveBedr Populat AveOccu Latitud Longitu MedHous
    MedInc █+1.00  ·-0.00  █+0.93  █+0.85  ·+0.01  ·-0.01  ·-0.02  ·+0.00  █+0.88  
   HouseAg ·-0.00  █+1.00  ·+0.00  ·-0.02  ·-0.00  ·+0.01  ·-0.03  ·-0.03  --0.11  
   AveRoom █+0.93  ·+0.00  █+1.00  █+0.91  ·+0.01  ·-0.01  ·-0.02  ·+0.01  █+0.83  
   AveBedr █+0.85  ·-0.02  █+0.91  █+1.00  ·+0.02  ·-0.02  ·-0.01  ·+0.01  █+0.75  
   Populat ·+0.01  ·-0.00  ·+0.01  ·+0.02  █+1.00  ·+0.04  ·+0.01  ·-0.01  --0.06  
   AveOccu ·-0.01  ·+0.01  ·-0.01  ·-0.02  ·+0.04  █+1.00  ·+0.00  ·-0.02  ·+0.00  
   Latitud ·-0.02  ·-0.03  ·-0.02  ·-0.01  ·+0.01  ·+0.00  █+1.00  ·-0.02  ░+0.24  
   Longitu ·+0.00  ·-0.03  ·+0.01  ·+0.01  ·-0.01  ·-0.02  ·-0.02  █+1.00  --0.13  
   MedHous █+0.88  --0.11  █+0.83  █+0.75  --0.06  ·+0.00  ░+0.24  --0.13  █+1.00  


In [4]:
# Zamieniamy nazwy kolumn na zwykłą listę, aby łatwo korzystać z indeksów.
labels = list(df.columns)

# Odczytujemy indeks targetu, który trzeba pominąć w rankingu.
target_idx = labels.index("MedHouseVal")

# W tej liście zapiszemy: wartość bezwzględną korelacji oraz indeksy i, j.
pairs = []

# Przechodzimy po parach leżących nad diagonalą macierzy.
# j zaczyna się od i + 1, więc pomijamy diagonalę i nie powtarzamy par.
for i in range(len(labels)):
    for j in range(i + 1, len(labels)):
        # Dodajemy parę tylko wtedy, gdy żadna cecha nie jest targetem.
        if i != target_idx and j != target_idx:
            pairs.append((np.abs(C[i, j]), i, j))

# Sortujemy malejąco - od największej bezwzględnej korelacji.
pairs.sort(reverse=True)

# Wybieramy pierwsze 3 elementy posortowanej listy.
top_3_pairs = pairs[:3]

# Wyświetlamy nazwy cech oraz właściwy współczynnik r ze znakiem.
print("3 pary cech o najwyższej korelacji bezwzględnej:")
for place, (_, i, j) in enumerate(top_3_pairs, start=1):
    print(f"{place}. {labels[i]} / {labels[j]}: r = {C[i, j]:+.4f}")

3 pary cech o najwyższej korelacji bezwzględnej:
1. MedInc / AveRooms: r = +0.9303
2. AveRooms / AveBedrms: r = +0.9056
3. MedInc / AveBedrms: r = +0.8451


### Interpretacja

1. **MedInc / AveRooms** – silna dodatnia korelacja wynika z tego, że w generatorze `AveRooms` zależy bezpośrednio od `MedInc`. Rejony o wyższych dochodach mają więc przeciętnie więcej pokoi.
2. **AveRooms / AveBedrms** – liczba sypialni jest częścią ogólnej liczby pokoi, a generator tworzy `AveBedrms` na podstawie `AveRooms`.
3. **MedInc / AveBedrms** – jest to głównie zależność pośrednia: `MedInc` wpływa na `AveRooms`, a `AveRooms` wpływa na `AveBedrms`.

> W tej syntetycznej wersji danych `Latitude` i `Longitude` są losowane niezależnie, dlatego nie tworzą jednej z najsilniej skorelowanych par. W rzeczywistych danych geograficznych taka zależność może być wyraźniejsza.

Korelacja opisuje związek liniowy między cechami, ale sama w sobie nie dowodzi związku przyczynowo-skutkowego.